# Amazon ML Challenge 2026 - Task 2

Training-only candidate generation, Recall@K evaluation, and retrieval failure analysis. The scalable implementation lives in `src/task2_candidate_generation.py`; this notebook provides a lightweight entry point and result review. It intentionally stops before pairwise feature engineering or classifier training.

In [ ]:
from pathlib import Path
import subprocess
import sys

import pandas as pd
try:
    from IPython.display import display
except ImportError:
    def display(value):
        print(value.to_string(index=False) if hasattr(value, 'to_string') else value)

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'student_resource').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / 'student_resource' / 'dataset' / 'train'
OUTPUT_DIR = PROJECT_ROOT / 'outputs' / 'task2_outputs'
SCRIPT_PATH = PROJECT_ROOT / 'src' / 'task2_candidate_generation.py'

print('Project root:', PROJECT_ROOT)
print('Training data:', DATA_DIR)
print('Task 2 outputs:', OUTPUT_DIR)

## Regenerate Task 2 outputs

Set `RUN_FULL_EXPERIMENT` to `True` only when the outputs need to be regenerated. The complete run evaluates 10,000 S1 entities against all 10.32 million S2/S3 candidates and takes about 14 minutes on the measured machine.

In [ ]:
RUN_FULL_EXPERIMENT = False

if RUN_FULL_EXPERIMENT:
    command = [
        sys.executable,
        str(SCRIPT_PATH),
        '--data-dir', str(DATA_DIR),
        '--output-dir', str(OUTPUT_DIR),
    ]
    subprocess.run(command, check=True, cwd=PROJECT_ROOT)
else:
    print('Using the existing Task 2 outputs.')

## Country validation and evaluation sample

In [ ]:
country_validation = pd.read_csv(OUTPUT_DIR / 'task2_country_validation.csv')
eval_entities = pd.read_csv(OUTPUT_DIR / 'task2_eval_entities.csv')

display(country_validation)
display(
    eval_entities.groupby('match_group', as_index=False)
    .agg(entities=('source1_entity_id', 'size'), positive_links=('match_count', 'sum'))
)

## Strategy comparison

In [ ]:
strategy_comparison = pd.read_csv(OUTPUT_DIR / 'task2_strategy_comparison.csv')
display(
    strategy_comparison[[
        'strategy', 'k', 'link_recall', 'complete_recall',
        'average_candidates', 'median_candidates', 'p95_candidates', 'max_candidates'
    ]].sort_values(['link_recall', 'average_candidates'], ascending=[False, True])
)

## Source, match-count, runtime, and failure analysis

In [ ]:
recall_by_source = pd.read_csv(OUTPUT_DIR / 'task2_recall_by_source.csv')
recall_by_match_count = pd.read_csv(OUTPUT_DIR / 'task2_recall_by_match_count.csv')
runtime_memory = pd.read_csv(OUTPUT_DIR / 'task2_runtime_memory.csv')
retrieval_failures = pd.read_csv(OUTPUT_DIR / 'task2_retrieval_failures.csv')
failure_patterns = pd.read_csv(OUTPUT_DIR / 'task2_failure_patterns.csv')

best = strategy_comparison.sort_values(
    ['link_recall', 'complete_recall', 'average_candidates'],
    ascending=[False, False, True],
).iloc[0]
best_mask_source = (
    recall_by_source['strategy'].eq(best['strategy'])
    & pd.to_numeric(recall_by_source['k'], errors='coerce').eq(float(best['k']))
)
best_mask_group = (
    recall_by_match_count['strategy'].eq(best['strategy'])
    & pd.to_numeric(recall_by_match_count['k'], errors='coerce').eq(float(best['k']))
)

display(recall_by_source.loc[best_mask_source])
display(recall_by_match_count.loc[best_mask_group])
display(runtime_memory)
display(failure_patterns)
display(retrieval_failures.head(50))

## Stop

Task 2 ends after candidate-generation evaluation and failure analysis. Do not proceed to pairwise classifier training until these results have been reviewed.